# Importação de bibliotecas 📚

In [15]:
from langchain_community.document_loaders import DirectoryLoader, BSHTMLLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from collections import Counter
from sys import exit
from textwrap import dedent
from IPython.display import display
import pandas as pd
import torch
import re
import pickle
import uuid
import inspect

# Sessão - Banco de Dados 🏦

## Objetivo Da Sessão 🧬:
* Criar um Banco de Dados Vetorial.

# Definindo a primeira classe do RAG 1️⃣.

## Principais objetivos 📝:
* Verificar se o usuário vai criar um novo Banco de Dados, se já criou, não será possível criar outro;
* Definir o modelo de Embedding;
* Carregar o Banco de Dados com Chroma;
* Verificar se o Banco de Dados está vazio;
* Pesquisar elementos do Banco de Dados.

In [2]:
class DBConnection:
    """The first class of Compass Rag Project"""

    _instance = None

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance._inicializate = False

        return cls._instance

    @property
    def embedding_model(self):
        """This method have the function to get the embedding model
        
        :returns the embedding model
        """
        
        device = "cuda" if torch.cuda.is_available() else "cpu"
        
        return HuggingFaceEmbeddings(
            model_name="BAAI/bge-base-en-v1.5",
            model_kwargs={'device': device}
        )

    def __init__(self, directory) -> None:
        """Main constructor of the class
        
        :param directory -> The directory to be given
        """
        
        if not self._instance._inicializate: 
            try:
                self.db = Chroma (
                    persist_directory = directory,
                    embedding_function = self.embedding_model,
                    collection_metadata={"hnsw:space": "cosine"}
                )
    
                self._instance._inicializate = True

                print("✅ \033[0;32mData Bank Sucessfully created!\033[m")

            except Exception as error:
                print(f"\033[0;31m🚨 Error: {error}\033[m")
                exit(1)

        else:
            print("⚠️ \033[1;31mAlready created a Data Bank.\033[m")
            return

    def db_is_empty(self) -> bool:
        """This method have the function to verify if is empty or not
        
        
        :returns -> True or False. It depends of the situation of DB's storage.
        """
        
        return len(self.db.get()["ids"]) == 0
        

    def search_information(self, question: str, filter: str = None, k=10) -> list:
        """This method have the function to search information on vetorized data bank
        
        :param question -> The question by the user to be given
        :param filter(opitional) -> The filter to help on search
        :param k -> The number of chunks to be showed

        :returns -> A list of chunks.
        """
        if not self.db_is_empty():
            available_titles = self.db.get().get("metadatas", [])
            unique_titles = list(set([m["title"] for m in available_titles if "title" in m]))

            instructional_query = f"Represent this sentence for searching relevant passages: {question}"
    
            search_kwargs = {
                "query": question,
                "k": k,
            }
    
            for title in unique_titles:

                if title.lower() in question.lower() or title.split(';')[0].lower() in question.lower():
                    search_kwargs["filter"] = {"title": title}
                    print(f"🤖 \033[0;34mAuto-Filter applied for:\033[m {title}")
                    break
    
            results = self.db.similarity_search_with_score(**search_kwargs)
            return results

        else:
            return "⚠️ \033[1;31mData bank is empty! It is not possible to search any information.\033[m"

# Definindo a segunda classe do RAG 2️⃣.

## Principais objetivos 📝:
* Carregar os arquivos em uma DataBase;
* Definir os chucks e overlaps;
* Filtragem de dados com o Regex;
* Adicionar esses novos dados ao Banco de Dados existente e vetorizar-lós;

In [3]:
class VectorizingData:
    """The second class of the Compass Rag Project"""

    def __init__(self, persistent_directory: str, db: DBConnection) -> None:
        """Main constructor of the class
        
        :param persistent_directory -> The directory to be given
        :param db -> The Data Bank Created
        """
        
        self.persistent_directory = persistent_directory
        self.db = db
        self._chunks = []


    def _clean_data(self, docs: list) -> list:
        """This method have the function of clear the data with regex and string manipulation

        :param docs -> The brute docs to be given
        :returns -> The clean data
        """
        clean_docs: list = []

        padroes_inicio = [
            r"\n\s*Letter [1I]\b",
            r"\n\s*Chapter [1I]\b",
            r"\n\s*CHAPTER ONE\b",
            r"\n\s*ACT [1I]\b",
            r"\n\s*PART [1I]\b",
            r"(?<=\n)\s*I\b\s*(?=\n)" 
        ]

        for doc in docs:
            brute_docs = doc.page_content

            title_match = re.search(r"(?<=Title:)(?P<title>.+)", brute_docs)
            if title_match:
                doc.metadata["title"] = title_match.group("title").strip()

            author_match = re.search(r"(?<=Author:)(?P<author>.+)", brute_docs)
            if author_match:
                doc.metadata["author"] = author_match.group("author").strip()

            data_match = re.search(r"(?<=Release\sdate:)(?P<data>.+?)(?=\s\[)", brute_docs)
            if data_match:
                doc.metadata["data"] = data_match.group("data").strip()

            language_match = re.search(r"(?<=Language:)(?P<language>.+)", brute_docs)
            if language_match:
                doc.metadata["language"] = language_match.group("language").strip()


            pos_i = brute_docs.find("*** START OF")
            pos_f = brute_docs.find("*** END OF")
            
            if pos_i != -1 and pos_f != -1:
 
                cleaned_text = brute_docs[brute_docs.find("\n", pos_i) : pos_f].strip()
                
            else:
                cleaned_text = brute_docs 

            for p in padroes_inicio:
                matches = list(re.finditer(p, cleaned_text, re.IGNORECASE | re.MULTILINE))

                if matches:
                    cleaned_text = cleaned_text[matches[-1].start():].strip()
                    break 

            cleaned_text = re.sub(
                r"THERE IS AN ILLUSTRATED EDITION.*?[\]]\s*", 
                "", 
                cleaned_text, 
                flags=re.IGNORECASE | re.DOTALL
            )

            doc.page_content = cleaned_text.strip()
            clean_docs.append(doc)

        return clean_docs
            

        
    def _loading_data(self) -> list | str:
        """This method have the function of loading the documents
        
        :returns -> The documents loaded
        """

        try:
            loader = DirectoryLoader(
                path = self.persistent_directory,
                glob = "*.html",
                loader_cls = BSHTMLLoader
            )
    
            brute_docs: list = loader.load()
            clean_docs = self._clean_data(brute_docs)

            return clean_docs

        except Exception as error:
            return f"\033[0;31m🚨 Error: {error}\033[m"


    def _data_chunk_and_overlap(self) -> list | str:
        """This method have the function of divide the content on chunks and do overlap between the chunks
        
        :returns -> The list of chunks getted
        """

        try:
            clean_docs = self._loading_data()
    
            chunk_splitter = RecursiveCharacterTextSplitter(
                chunk_size = 1200,
                chunk_overlap = 400
            )
    
            self._chunks = chunk_splitter.split_documents(clean_docs)

            for chunk in self._chunks:
                title = chunk.metadata.get('title', 'Unknown Book')
                
                chunk.page_content = f"Book Title: {title} | Chapter context: {chunk.page_content}"

                chunk.metadata["id"] = str(uuid.uuid4())

            return self._chunks
        
        except Exception as error:
            return f"\033[0;31m🚨 Error: {error}\033[m"


    @property
    def chunk_data_information(self) -> dict:
        """This method have the function of show some informations about the chunks
        
        :returns -> The information collected about the chunks
        """

        if len(self._chunks) == 0:
            return "⚠️ \033[1;31mYou don't have processed any chunks. Execute the method '_data_chunk_and_overlap' to work!\033[m"

        else:
            stats: dict = {
                "total_chunks": len(self._chunks),
                "per_title": {},
                "per_author": {},
                "per_language": {},
                "per_data": {},
            }

            for chunk in self._chunks:
                t = chunk.metadata.get('title', '?')
                a = chunk.metadata.get('author', '?')
                l = chunk.metadata.get('language', '?')
                d = chunk.metadata.get('data', '?')

                stats["per_title"][t] = stats["per_title"].get(t, 0) + 1
                stats["per_author"][a] = stats["per_author"].get(a, 0) + 1
                stats["per_language"][l] = stats["per_language"].get(l, 0) + 1
                stats["per_data"][d] = stats["per_data"].get(d, 0) + 1

            return stats


    def vectorizing_data(self):
        """Adds documents to ChromaDB using safe batching"""
        
        batch_size = 1000
        total = len(self._chunks)
        
        try:
            for i in range(0, total, batch_size):
                batch = self._chunks[i : i + batch_size]
                self.db.db.add_documents(batch)
                print(f"✅ Progress: {min(i + batch_size, total)}/{total} chunks vectorized.")
            return "🏆 \033[0;32mVectorization Complete!\033[m"
            
        except Exception as e:
            return f"\033[0;31m🚨 Batching Error: {e}\033[m"


    def save_to_pickle(self, filename="processed_chunks.pkl"):
        """PDF Requirement: Saves the processed data structure into a Pickle file"""
        
        serialized_data = []
        
        for chunk in self._chunks:
            serialized_data.append({
                "id": chunk.metadata.get("id"),
                "content": chunk.page_content,
                "metadata": chunk.metadata
            })
            
        with open(filename, "wb") as f:
            pickle.dump(serialized_data, f)
        print(f"📦 \033[0;33mData successfully serialized to {filename}\033[m")

# Sessão de Testes 📔

## Objetivos da sessão 🧬:
* Mostrar na prática o Embedding Trabalhando;
* Documentação de toda a minha trajetória para a realização do Desafio.

## Definindo os caminhos das pastas e das classes
- Banco de Dados
- Base de Dados

In [4]:
DATA_PATH = "data_base"
DB_PATH = "data_bank"

db_instance = DBConnection(DB_PATH)
vectorizing = VectorizingData(DATA_PATH, db_instance)

C:\Users\netoc\AppData\Local\Temp\ipykernel_1896\502012640.py:22: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(
Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 7492.79it/s]


✅ Data Bank Sucessfully created!


## Vetorizando o Banco de Dados 🏦
- Colocando os chunks em arquivos .pkl
- Aplicando os chunks e overlaps nos documentos HTML.
- Aplicando os chunks diretamente no Banco de Dados. Por conseguinte, vetorizando esses chunks.

### Processo Lógico 🧠
- O método "_data_chunk_and_overlap" vai chamar a função "_loading_data", vai pegar os dados que já foram limpos e vai amarzená-los para serem divididos em uma quantidade considerável de chunks e ligam os mesmos com overlaps. Contudo definimos um novo padrão para o conteúdo. Logo, todo chunk será acompanhado com o metadado "title". Outrossim, chamo o método "vectorizing_data" para guardar esses chunks dentro do banco de dados vetorizado e com a medida de segurança de 1000 batchs por vez para não estourar. Por fim, congelamos tudo e colocamos tudo em um arquivo .pkl.

In [5]:
vectorizing._data_chunk_and_overlap()
vectorizing.vectorizing_data()
vectorizing.save_to_pickle()

✅ Progress: 1000/6742 chunks vectorized.
✅ Progress: 2000/6742 chunks vectorized.
✅ Progress: 3000/6742 chunks vectorized.
✅ Progress: 4000/6742 chunks vectorized.
✅ Progress: 5000/6742 chunks vectorized.
✅ Progress: 6000/6742 chunks vectorized.
✅ Progress: 6742/6742 chunks vectorized.
📦 Data successfully serialized to processed_chunks.pkl


## Testando o Modelo 📳
- Digitação da pergunta de acordo com os documentos;
- Visualização dos 10 chunks para a pergunta;
- Visualização do score dos chunks;
- Visualização da porcentagem de confiabilidade dos chunks.

## Demonstração Prática (Gerada por IA) 🤖
- Demonstração de como vai ser retornado as chunks de acordo com as perguntas.

In [11]:
easy_questions = {
    "In Romeo and Juliet, what is the name of Juliet’s nurse?": "Romeo and Juliet",
    "What is Sherlock Holmes’s address?": "The Adventures of Sherlock Holmes",
    "How many Bennet sisters are there in Pride and Prejudice?": "Pride and Prejudice",
    "What is the name of the lawyer in Dr. Jekyll and Mr. Hyde?": "The strange case of Dr. Jekyll and Mr. Hyde",
    "Who is the landlord of Thrushcross Grange in Wuthering Heights?": "Wuthering Heights",
    "Who painted the portrait of Dorian Gray?": "The Picture of Dorian Gray",
    "Which family does Juliet belong to in Romeo and Juliet?": "Romeo and Juliet",
    "Who is Sherlock Holmes's loyal friend and biographer?": "The Adventures of Sherlock Holmes",
    "What is the name of the estate owned by Mr. Darcy?": "Pride and Prejudice",
    "What is the name of the actress Dorian Gray falls in love with?": "The Picture of Dorian Gray"
}

complex_questions = {
    "Who has a double life and hides their true identity?": "The strange case of Dr. Jekyll and Mr. Hyde",
    "Which works use isolated settings to reflect psychological states?": "Wuthering Heights",
    "Which characters undergo significant moral transformation?": "The Picture of Dorian Gray",
    "Which story focuses on the social expectations of marriage and class differences in 19th century England?": "Pride and Prejudice",
    "Which tragedy explores the catastrophic consequences of ancient family feuds on young lovers?": "Romeo and Juliet",
    "Which narrative relies heavily on deductive reasoning and forensic science to solve mysteries?": "The Adventures of Sherlock Holmes",
    "Which novel explores the terrifying consequences of eternal youth and vanity?": "The Picture of Dorian Gray",
    "Which story features a deeply destructive and obsessive romance spanning across generations on the moors?": "Wuthering Heights",
    "Which book deals with a scientific experiment gone wrong that unleashes humanity's dark side?": "The strange case of Dr. Jekyll and Mr. Hyde",
    "In which story does the protagonist's initial bad impression of a wealthy man slowly turn into profound love and respect?": "Pride and Prejudice"
}

def evaluate_compass_rag_top10(questions_dict, test_level):
    """This method have the function of give the results inspirated on questions given
    
    :param questions_dict -> The dict of questions given
    :param question_dict - > The level of test
    """
    print(f"\n\033[1;35m{'='*25} TEST BATCH: {test_level} (Top 10) {'='*25}\033[m")
    
    hits = 0
    total = len(questions_dict)

    for query, expected_book in questions_dict.items():
        print(f"\n\033[1;33m❓ Question:\033[m {query}")
        
        results = db_instance.search_information(query, k=10)
        
        if isinstance(results, str):
            print(results)
            continue
            
        if isinstance(results, list) and len(results) > 0:
            titles_found = [doc.metadata.get('title', 'Unknown Book') for doc, dist in results]
            
            is_hit = any(expected_book.lower() in t.lower() for t in titles_found)
            
            if is_hit:
                hit_idx = next(i for i, t in enumerate(titles_found) if expected_book.lower() in t.lower())
                correct_doc, correct_dist = results[hit_idx]
                
                similarity_score = 1 - correct_dist
                
                status = f"\033[1;32m✅ HIT (Position {hit_idx + 1} of Top 10)\033[m"
                hits += 1
                display_title = correct_doc.metadata.get('title')
                excerpt = correct_doc.page_content.replace('\n', ' ')
                metrics = f"Distance: {correct_dist:.4f} | \033[1;32mScore: {similarity_score:.4f}\033[m (Max: 1, Min: -1)"
                
            else:
                top_doc, top_dist = results[0]
                similarity_score = 1 - top_dist
                
                status = "\033[1;31m❌ MISS (Outside Top 10)\033[m"
                display_title = top_doc.metadata.get('title')
                excerpt = top_doc.page_content.replace('\n', ' ')
                metrics = f"Distance: {top_dist:.4f} | \033[1;31mScore: {similarity_score:.4f}\033[m (Max: 1, Min: -1)"

            print(f"Status      : {status}")
            print(f"Expected    : {expected_book}")
            print(f"Matched     : \033[1;36m{display_title}\033[m")
            print(f"Metrics     : {metrics}")
            print(f"Excerpt     : \"{excerpt[:150]}...\"")
        else:
            print("\033[1;31m⚠️ No results found.\033[m")
            
    accuracy = (hits / total) * 100
    score_color = "\033[1;32m" if accuracy == 100 else "\033[1;33m" if accuracy >= 80 else "\033[1;31m"
    print(f"\n{score_color}📊 FINAL SCORE ({test_level}): {hits}/{total} ({accuracy:.1f}%)\033[m")
    print(f"\033[1;35m{'='*90}\033[m")

evaluate_compass_rag_top10(easy_questions, "EASY QUESTIONS (FACTUAL)")
evaluate_compass_rag_top10(complex_questions, "COMPLEX QUESTIONS (SEMANTIC)")


========================= TEST BATCH: EASY QUESTIONS (FACTUAL) (Top 10) =========================

❓ Question: In Romeo and Juliet, what is the name of Juliet’s nurse?
🤖 Auto-Filter applied for: Romeo and Juliet
Status      : ✅ HIT (Position 1 of Top 10)
Expected    : Romeo and Juliet
Matched     : Romeo and Juliet
Metrics     : Distance: 0.3138 | Score: 0.6862 (Max: 1, Min: -1)
Excerpt     : "Book Title: Romeo and Juliet | Chapter context: [Exeunt.]  SCENE III. Juliet’s Chamber. Enter Juliet and Nurse.  JULIET. Ay, those attires are best. B..."

❓ Question: What is Sherlock Holmes’s address?
Status      : ✅ HIT (Position 1 of Top 10)
Expected    : The Adventures of Sherlock Holmes
Matched     : The Adventures of Sherlock Holmes
Metrics     : Distance: 0.3120 | Score: 0.6880 (Max: 1, Min: -1)
Excerpt     : "Book Title: The Adventures of Sherlock Holmes | Chapter context: “What office?”   “That’s the worst of it, Mr. Holmes, I don’t know.”   “Where did he ..."

❓ Question: How many Ben

#### Se desejar colocar mais perguntas ou novas é só colocar no dicionário de "easy_questions" ou "complex_questions" e testar os scores. 📔

## Processando arquivo .pkl 👾
- Demonstrar as informações contidas no arquivo .pkl


In [14]:
file_path = 'processed_chunks.pkl'

try:
    with open(file_path, 'rb') as f:
        chunks = pickle.load(f)

    organized_list = []

    for i, item in enumerate(chunks):
        text_data = str(item)
        
        title_match = re.search(r"Book Title:\s*(.*?)\s*\|", text_data)
        book_title = title_match.group(1) if title_match else "Unknown"
        
        content_match = re.search(r"Chapter context:\s*(.*)", text_data)
        clean_content = content_match.group(1) if content_match else text_data[:100]

        organized_list.append({
            "Chunk ID": i,
            "Book Title": book_title,
            "Char Count": len(clean_content),
            "Content Preview": clean_content[:80].replace("'", "").strip() + "..."
        })

    df = pd.DataFrame(organized_list)

    print(f"\n\033[1;35m{'='*20} ORGANIZED CHUNKS REPORT {'='*20}\033[m")
    
    summary = df.groupby('Book Title').size().reset_index(name='Total Chunks')
    display(summary)

    print("\n🔍 DATA PREVIEW (Corrected Titles):")
    display(df.head(15))

except Exception as e:
    print(f"❌ Error during parsing: {e}")


==================== ORGANIZED CHUNKS REPORT ====================


,Book Title,Total Chunks
0,A Room with a View,456
1,Adventures of Huckleberry Finn,698
2,Alice's Adventures in Wonderland,174
3,"Frankenstein; or, the modern prometheus",530
4,Pride and Prejudice,948
5,Romeo and Juliet,170
6,The Adventures of Sherlock Holmes,709
7,The Blue Castle: a novel,484
8,The Enchanted April,535
9,The Great Gatsby,331



🔍 DATA PREVIEW (Corrected Titles):


,Chunk ID,Book Title,Char Count,Content Preview
0,0,A Room with a View,1288,"Chapter I\nThe Bertolini\n\n“The Signora had no business to do it,” said Mis..."
1,1,A Room with a View,1423,"“This meat has surely been used for soup,” said Miss Bartlett, laying down h..."
2,2,A Room with a View,1401,"“She would never forgive me.”\n\n\nThe ladies’ voices grew animated, and—if ..."
3,3,A Room with a View,1175,"“This is my son,” said the old man; “his name’s George. He has a view too.”\..."
4,4,A Room with a View,1156,"“You see, we don’t like to take—” began Lucy. Her cousin again repressed\nhe..."
5,5,A Room with a View,1420,"Miss Bartlett, though skilled in the delicacies of conversation, was powerle..."
6,6,A Room with a View,1291,"Miss Bartlett said, with more restraint:\n\n\n“How do you do, Mr. Beebe? I e..."
7,7,A Room with a View,1043,"“Miss Honeychurch lives in the parish of Summer Street,” said Miss Bartlett,..."
8,8,A Room with a View,1242,"“Oh, how glad I am! The name of our house is Windy Corner.” Mr. Beebe bowed...."
9,9,A Room with a View,1427,"“No!” cried a voice from the top of the table. “Mr. Beebe, you are wrong. Th..."


# Considerações Finais:
**_"O trabalho do RAG foi muito trabalhoso de ser feito. Os instrutores disseram que iriamos rir de tanto que seria fácil. Bom, eu ri, não pela dificuldade e sim pela jornada de aprendizado que eu iria desenvolver fazendo esse desafio proposto pela Compass. Foi uma série de testes de modelos de embeddings para ver quem tinha a melhor perfomance e acabei escolhendo esse para o desafio. Utilizei muito regex para filtrar os cabeçários e os rodapés dos livros e deixando apenas a parte de conteúdo como importante. Mesmo assim, ainda o do Sherlock Homes não consegui filtrar o sumário de capítulos por ser um modelo diferente dos demais. Sinto-me honrado em participar dessa jornada com vocês e espero está presente na próxima fase. Tenho muito que aprender com cada um e seguir o meu sonho de se tornar o melhor Engenheiro de Machine Learning. Mesmo com esses transtornos que tenho como autismo e um TDAH sutil, nada me impede ou vai me derrubar em questão aprendizagem, pois sou apaixonado pelo o que eu faço."_**